In [1]:
#----------------------------------------------------------------------------------------------------------------------------#
#
# Code:      CC01_CCAR_D03_Deterministic_SQL_Generation_01.ipynb
#
# Objective: Step DO3: Develop some Deterministic SQL Generation tool, which is used to improve risk goverance of Agentic AI.
#
#            Jingru Chen
#            2026-04-02
#
#-----------------------------------------------------------------------------------------------------------------------------#

In [2]:
#       Purpose:
#
#       The Agentic Semantic Layer: Fixing the 'Context Gap' in Banking
#       (1) The Problem: LLMs are not 100% Reliable.
#           LLMs are great at understanding what a person wants, but they are "probabilistic." This means they guess the next word.
#           In banking (like CCAR reporting, Risk Management, Compliance, etc.), being "mostly right" is a failure.
#           We cannot have an LLM "guessing" a SQL query because it might hallucinate a table name or a calculation. This creates a
#           "Context Gap" between the user's question and the real data.
#
#       (2) The Solution: Splitting 'Thinking' from 'Doing'
#           I built a system that separates Intent (what the user wants) from Execution (running the code). Let's call this the Agentic Semantic Layer.
#           In this system, the LLM only handles the language, while a rigid Python program handles the math and the SQL.

#           (2-A). The LLM handles the "Intent"
#           The LLM only does one job: Translation. It takes a human question like "Show me the loal balance of the vintage of 202401" and turns it into a simple JSON file.
#           It does not write the SQL. It only identifies the "parameters" of the request.
#
#           (2-B) The Python Compiler handles the "Code"
#           A Python program (the "Compiler") reads that JSON file. It uses a fixed Semantic Layer (a list of real business rules) to create the SQL query.
#           This is "deterministic," meaning it works the same way every single time. This method ensures:
#
#           Real Rules: We use the actual bank rules, not the LLM's memory.
#           Security: The LLM cannot "make up" table names.
#           Audit: Every SQL query can be checked and saved for regulators.
#
#       (3) The Result: Bank-Ready Data
#           Using CCAR data, I proved that we can give analysts a "chat" experience that is still safe and accurate.
#           This meets the SR 11-7 rules because the logic is transparent and reproducible. We closed the gap between human questions and the bank's database.
#

# Step 0: Upload libraries

In [3]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from pydantic import BaseModel, Field

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
ls -ltr

total 8
drwxr-xr-x 1 root root 4096 Mar 30 13:29 sample_data/
drwx------ 5 root root 4096 Apr  3 00:39 drive/


In [6]:
# cat config.py

In [7]:
cd /content/drive/MyDrive/Colab Notebooks

/content/drive/MyDrive/Colab Notebooks


In [8]:
ls -ltr

total 30133
-rw------- 1 root root   994796 Jan 13  2025  G01_learn_run_code_in_Google_CoLab_20250102.ipynb
-rw------- 1 root root    47208 Mar 10 05:44  S01_NLP_preprocessing_traditional_Machine_Learning_01.ipynb
-rw------- 1 root root   119835 Mar 10 07:47  S02_NLP_BERT_FlanT5_models_01.ipynb
-rw------- 1 root root    36703 Mar 12 04:43  AA_001_Test_Agent_AI_workflow_for_Stock_prices_01_20260312.ipynb
-rw------- 1 root root    87341 Mar 12 04:55  AA_001_Test_Agent_AI_workflow_for_Stock_prices_02_20260312.ipynb
-rw------- 1 root root     6633 Mar 16 14:20  Untitled0.ipynb
-rw------- 1 root root     1634 Mar 16 14:23  Untitled1.ipynb
-rw------- 1 root root     6172 Mar 16 17:57  S01_How_to_fit_a_logit_model_in_Python_20260316.ipynb
-rw------- 1 root root    21099 Mar 18 18:26  AA_002_Test_Agent_AI_workflow_for_CCAR_CECL_01_20260318.ipynb
-rw------- 1 root root 29210174 Mar 20 01:38  CCAR_Mortgage_data_for_model_DEV_20260319_01.csv
-rw------- 1 root root     2209 Mar 22 17:28  ccar_pd_m

In [9]:
import sys
from google.colab import drive

# 1. Mount the drive
drive.mount('/content/drive')

# 2. Add the specific folder to Python's search path
# Note the space in "Colab Notebooks"
path_to_config = '/content/drive/MyDrive/Colab Notebooks'
if path_to_config not in sys.path:
    sys.path.append(path_to_config)

# 3. Now you can import normally
from config import MY_API_KEY

print(f"Success! My OpenAI API Key loaded: {MY_API_KEY[:20]}****")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Success! My OpenAI API Key loaded: sk-proj-fezFts2agCx8****


In [10]:
from datetime import datetime
from zoneinfo import ZoneInfo

run_date="2026-04-02"

start = datetime.now( ZoneInfo("America/New_York"))

print( start.strftime("%Y-%m-%d %H:%M:%S %Z"))     # 2026-03-17 17:34:58 EDT
print( start.strftime("%Y-%m-%d %I:%M:%S %p %Z"))  # 2026-03-17 05:34:58 PM EDT

2026-04-02 22:25:35 EDT
2026-04-02 10:25:35 PM EDT


In [11]:
x_list=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate', 'loan_term_months',
        'delta_Unemployment1',
       'delta_Mortgage_rate1', 'delta_House_Price_Index__Level1',
       'delta_Unemployment3', 'delta_Mortgage_rate3',
       'delta_House_Price_Index__Level3', 'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12', 'delta_Mortgage_rate12',
       'delta_House_Price_Index__Level12', 'delta_Unemployment24',
       'delta_Mortgage_rate24', 'delta_House_Price_Index__Level24']

x_list_v2=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate',
        'delta_Unemployment1',
       'delta_Unemployment3',
       'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12',
       'delta_Unemployment24' ]

# y_list= ['flag_default']

pd_model = 'ccar_pd_model_2026-03-22.pkl'

# pd_input_file= "/CCAR_Mortgage_data_for_model_DEV_20260319_01.csv"
# Use the full Absolute Path:
pd_input_file = "/content/drive/MyDrive/Colab Notebooks/CCAR_Mortgage_data_for_model_DEV_20260319_01.csv"

# Step 1: Upload my simulated CCAR file

In [12]:
df = pd.read_csv(pd_input_file)
df['report_yrmo'] = df['report_yrmo'].astype(str)

print( df.columns)

print( df.product_type.value_counts() )
# print( df.report_yrmo.value_counts() )

df_short = df[['loan_id', 'product_type', 'report_yrmo', 'original_balance', 'current_balance', 'credit_score_orig']].loc[df['report_yrmo'] <= '201412']

df_short.info()

Index(['Unnamed: 0', 'loan_id', 'origination_date', 'report_date',
       'num_payments', 'original_balance', 'current_balance_bk', 'EAD',
       'credit_score_orig', 'loan_to_value_orig', 'interest_rate',
       'product_type', 'ever_defaulted', 'default_date', 'PD', 'LGD',
       'projected_loss', 'loan_term_months', 'unemployment', 'gdp_growth_qoq',
       'hpi_change', 'bbb_spread', 'months_elapsed', 'current_balance', 'yrmo',
       'report_yrmo', 'default_yrmo', 'flag_default', 'flag_removal',
       'delta_Unemployment1', 'delta_Mortgage_rate1',
       'delta_House_Price_Index__Level1', 'delta_Unemployment3',
       'delta_Mortgage_rate3', 'delta_House_Price_Index__Level3',
       'delta_Unemployment6', 'delta_Mortgage_rate6',
       'delta_House_Price_Index__Level6', 'delta_Unemployment12',
       'delta_Mortgage_rate12', 'delta_House_Price_Index__Level12',
       'delta_Unemployment24', 'delta_Mortgage_rate24',
       'delta_House_Price_Index__Level24', 'flag_merge'],
      dt

# Step 2: Create a simple example of Deterministic SQL Generation without calling LLM

### Step 2-A: Upload simulated CCAR file

In [13]:
import sqlite3
import pandas as pd

# Create an in-memory database for our CCAR simulated data
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Table 1: Loan Portfolio Results (Stressed)
cursor.execute('''
CREATE TABLE FACT_CCAR_RESULTS (
    loan_id      TEXT,
    product_type TEXT,
    report_yrmo  TEXT,
    original_balance REAL,
    current_balance REAL,
    credit_score_orig REAL
)''')


# Convert dataframe to list of lists (excluding the index)
data_to_insert = df_short.values.tolist()

cursor.executemany('INSERT INTO FACT_CCAR_RESULTS VALUES (?,?,?,?,?,?)', data_to_insert )

conn.commit()

### Step 2-B: Set up Semantic Layer and Deterministic SQL Generator without calling LLM

In [14]:
# --- The SEMANTIC Map (The "Governed Truth") ---
SEMANTIC_MAP = {
    "entities": {
        "portfolio": "FACT_CCAR_RESULTS"
    },
    "metrics": {
        "total_original_balance": "SUM(original_balance)",
        "total_current_balance": "SUM(current_balance)",
        "avg_credit_score": "SUM(credit_score_orig) / COUNT(credit_score_orig) * 100"
    },
    "dimensions": {
        "type": "product_type",
        "vintage": "report_yrmo"
    }
}


class CcarDeterministicGenerator:
    def __init__(self, semantic_map):
        self.map = semantic_map

    def generate_and_execute(self, intent, db_conn):
        # 1. Map logical intent to physical schema
        table = self.map["entities"].get(intent["entity"])
        metric_sql = self.map["metrics"].get(intent["metric"])
        dim_sql = self.map["dimensions"].get(intent["group_by"])
        filter_col = self.map["dimensions"].get(intent["filter_key"])
        filter_val = intent["filter_val"]

        # 2. Compile the Deterministic SQL
        sql = f"""
        SELECT {dim_sql}, {metric_sql} AS result
        FROM {table}
        WHERE {filter_col} = '{filter_val}'
        GROUP BY {dim_sql}
        """

        # 3. Execute and return results
        print(f"--- LOG: Executing Governed SQL ---\n{sql}")
        return pd.read_sql_query(sql, db_conn)

# --- EXECUTION EXAMPLE ---
# Imagine an Agent parses a user's question:
# "What is the total original balance for HELOC?"

agent_intent = {
    "entity": "portfolio",
    "metric": "total_original_balance",
    "group_by": "vintage",
    "filter_key": "type",
    "filter_val": "HELOC"
}

generator = CcarDeterministicGenerator(SEMANTIC_MAP)
results = generator.generate_and_execute(agent_intent, conn)

print("\n--- FINAL CCAR RESULT of HELOC ---")
print(results)

--- LOG: Executing Governed SQL ---

        SELECT report_yrmo, SUM(original_balance) AS result
        FROM FACT_CCAR_RESULTS
        WHERE product_type = 'HELOC'
        GROUP BY report_yrmo
        

--- FINAL CCAR RESULT of HELOC ---
   report_yrmo       result
0       201401  76469616.51
1       201402  76469616.51
2       201403  75862106.81
3       201404  73929139.13
4       201405  71047064.94
5       201406  68334015.40
6       201407  66662409.98
7       201408  64689148.58
8       201409  63173722.56
9       201410  60630284.33
10      201411  59350053.91
11      201412  58607025.95


# Step 3: Create an example of Deterministic SQL Generation via calling LLM

### Step 3-A: Creating a vintage-level Original-loan-balance summary for HELOC

In [15]:
import os
import json
from openai import OpenAI

# 1. Initialize the client (ensure that I can access some pre-trained LLM)
# User my current OpenAI API Key
client = OpenAI(api_key=MY_API_KEY)
print( client )

def parse_user_question(question):
    """
    Uses OpenAI's Function Calling to force the LLM into a deterministic JSON format.
    """

    # Define the "Form" the LLM must fill out
    ccar_tool = {
        "type": "function",
        "function": {
            "name": "generate_ccar_intent",
            "description": "Extracts CCAR query parameters from natural language.",
            "parameters": {
                "type": "object",
                "properties": {
                    "entity": {"type": "string", "enum": ["portfolio"]},
                    "metric": {"type": "string", "enum": ["total_original_balance", "total_current_balance", "avg_credit_score"]},
                    "group_by": {"type": "string", "enum": ["vintage"]},
                    "filter_key": {"type": "string", "enum": ["type"]},
                    "filter_val": {"type": "string", "description": "The specific product name, e.g., 'HELOC'"}
                },
                "required": ["entity", "metric", "group_by", "filter_key", "filter_val"]
            }
        }
    }


    # 2. Call the LLM
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a specialized CCAR data architect. Map user questions to the tool provided."},
            {"role": "user", "content": question}
        ],
        tools=[ccar_tool],
        tool_choice={"type": "function", "function": {"name": "generate_ccar_intent"}}
    )

    # 3. Extract the structured JSON from the tool call
    tool_call = response.choices[0].message.tool_calls[0]
    intent_json = json.loads(tool_call.function.arguments)

    return intent_json

# --- Execution ---
user_query = "please let me know the total original balance for HELOC"
agent_intent = parse_user_question(user_query)

# --- The SEMANTIC Map (The "Governed Truth") ---
SEMANTIC_MAP = {
    "entities": {
        "portfolio": "FACT_CCAR_RESULTS"
    },
    "metrics": {
        "total_original_balance": "SUM(original_balance)",
        "total_current_balance": "SUM(current_balance)",
        "avg_credit_score": "SUM(credit_score_orig) / COUNT(credit_score_orig) * 100"
    },
    "dimensions": {
        "type": "product_type",
        "vintage": "report_yrmo"
    }
}

# Initialize the generator with the map
generator = CcarDeterministicGenerator(SEMANTIC_MAP)

# Run the query against my simulated CCAR database connection 'conn'
results = generator.generate_and_execute(agent_intent, conn)

print("\n--- Final CCAR Result of vintage-level HELOC ---")
print(results)


--- LOG: Executing Governed SQL ---

        SELECT report_yrmo, SUM(original_balance) AS result
        FROM FACT_CCAR_RESULTS
        WHERE product_type = 'HELOC'
        GROUP BY report_yrmo
        

--- Final CCAR Result of vintage-level HELOC ---
   report_yrmo       result
0       201401  76469616.51
1       201402  76469616.51
2       201403  75862106.81
3       201404  73929139.13
4       201405  71047064.94
5       201406  68334015.40
6       201407  66662409.98
7       201408  64689148.58
8       201409  63173722.56
9       201410  60630284.33
10      201411  59350053.91
11      201412  58607025.95


### Step 3-B: Creating a vintage-level Original-loan-balance summary for Mortgage

In [16]:
# --- Execution ---
user_query = "what is the total original loan balance for Mortgage?"
agent_intent = parse_user_question(user_query)

# --- The SEMANTIC Map (The "Governed Truth") ---
SEMANTIC_MAP = {
    "entities": {
        "portfolio": "FACT_CCAR_RESULTS"
    },
    "metrics": {
        "total_original_balance": "SUM(original_balance)",
        "total_current_balance": "SUM(current_balance)",
        "avg_credit_score": "SUM(credit_score_orig) / COUNT(credit_score_orig) * 100"
    },
    "dimensions": {
        "type": "product_type",
        "vintage": "report_yrmo"
    }
}

# Initialize the generator with the map
generator = CcarDeterministicGenerator(SEMANTIC_MAP)

# Run the query against my simulated CCAR database connection 'conn'
results = generator.generate_and_execute(agent_intent, conn)

print("\n--- 3.B Final CCAR Result of vintage-level Mortgage ---")
print(results)


--- LOG: Executing Governed SQL ---

        SELECT report_yrmo, SUM(original_balance) AS result
        FROM FACT_CCAR_RESULTS
        WHERE product_type = 'Mortgage'
        GROUP BY report_yrmo
        

--- 3.B Final CCAR Result of vintage-level Mortgage ---
   report_yrmo        result
0       201401  3.518960e+08
1       201402  3.518960e+08
2       201403  3.437731e+08
3       201404  3.350509e+08
4       201405  3.256945e+08
5       201406  3.131489e+08
6       201407  3.033729e+08
7       201408  2.925174e+08
8       201409  2.838582e+08
9       201410  2.752680e+08
10      201411  2.697081e+08
11      201412  2.619925e+08


In [17]:
from datetime import datetime
end = datetime.now(ZoneInfo("America/New_York"))
duration = end - start

print(f"Started:  {start}")
print(f"Finished: {end}")
print(f"\nDuration: {duration}")                    # 0:00:02.351234
print(f"Duration: {duration.total_seconds():.3f} seconds")

Started:  2026-04-02 22:25:35.215483-04:00
Finished: 2026-04-02 22:25:43.291623-04:00

Duration: 0:00:08.076140
Duration: 8.076 seconds
